# **TrustJob KZ - Recommendation Module**
## Similar Jobs · Resume Matching · Skill Gap Analysis

---

**Module:** `recommendation_engine.py`  
**Safe job database:** Kazakhstan jobs dataset  

| Feature | Method | Output |
|---|---|---|
| Similar real job recommendations | TF-IDF + Cosine Similarity | Top-N safe jobs |
| Resume ↔ Job matching | TF-IDF + Cosine Similarity | Top-N jobs by resume fit |
| Skill gap analysis | Keyword set difference | Missing skills + match % |

> All recommendations come **only from verified safe jobs**.


---
## **1. Import Libraries**

In [1]:
import os, re, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs('models', exist_ok=True)
print('Libraries loaded.')


Libraries loaded.


---
## **2. Academic Context**

### Why TF-IDF + Cosine Similarity?

**TF-IDF** converts job text into a numeric vector. Words that are frequent in one job but rare across all jobs receive higher weight — capturing what makes each posting unique.

**Cosine similarity** measures the angle between two vectors regardless of length:

$$\text{similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

Score of 1.0 = identical content. Score of 0.0 = no shared vocabulary. Preferred over Euclidean distance for text because document length does not distort results.

### Why only safe jobs as recommendations?

Recommending a fraudulent job defeats the purpose of TrustJob KZ. The recommendation index is built **exclusively from real/safe jobs**.

### Safe job filtering logic

| Scenario | Filter |
|---|---|
| Dataset has `fraudulent` / `is_fake` label | Keep label = 0 |
| Dataset has risk scores from our model | Keep `risk_level` ≠ "High" |
| No labels available | Use all jobs (fallback) with warning |


---
## **3. Load Kazakhstan Dataset**

In [2]:
def find_existing_path(possible_paths):
    for path in possible_paths:
        if os.path.exists(path):
            return path
    return None


def read_jobs_csv(path, preferred_sep=None):
    if preferred_sep is not None:
        return pd.read_csv(path, sep=preferred_sep, quoting=3, on_bad_lines='skip', engine='python', encoding='utf-8')
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        sample = f.read(4096)
    sep = ';' if sample.count(';') > sample.count(',') else ','
    return pd.read_csv(path, sep=sep, quoting=3, on_bad_lines='skip', engine='python', encoding='utf-8')

KZ_DATA_PATH = '/Users/aruzhantleukul/Desktop/machine learning/TrustJob/kazakhstan_jobs.csv'

raw_kz = read_jobs_csv(KZ_DATA_PATH)
DATA_SOURCE = '/Users/aruzhantleukul/Desktop/machine learning/TrustJob/kazakhstan_jobs.csv'

print(f'Kazakhstan dataset path: {KZ_DATA_PATH}')
print(f'Kazakhstan dataset: {raw_kz.shape[0]:,} rows')
print('Columns:', raw_kz.columns.tolist())
raw_kz.head(3)

Kazakhstan dataset path: /Users/aruzhantleukul/Desktop/machine learning/TrustJob/kazakhstan_jobs.csv
Kazakhstan dataset: 636 rows
Columns: ['title', 'company_name', 'has_company_name', 'salary', 'has_salary', 'url', 'source', 'description', 'label', 'id', 'text']


,title,company_name,has_company_name,salary,has_salary,url,source,description,label,id,text
0,Требуется водитель для экскаватора погрузчика ...,NaN,0,NaN,0,https://www.olx.kz/obyavlenie/rabota/trebuetsy...,olx.kz,NaN,1,458,требуется водитель для экскаватора погрузчика ...
1,Администратор в студию лазерной эпиляции,ТОО Лала,1,NaN,0,https://almaty.hh.kz/vacancy/131831768?hhtmFro...,hh.kz,👩‍💼 администратор студии лазерной эпиляции | в...,0,281,администратор в студию лазерной эпиляции админ...
2,Оператор call-центра,ТОО Медицинский центр EMIRMED,1,"от 200 000 до 350 000 ₸ за месяц, на руки",1,https://almaty.hh.kz/vacancy/131619531?hhtmFro...,hh.kz,многопрофильные медицинские центры эмирмед уже...,0,276,оператор call центра многопрофильные медицинск...


---
## **4. Standardise & Filter Safe Jobs**

In [3]:
KZ_COL_MAP = {
    'job_title': 'title', 'vacancy': 'title', 'name': 'title', 'job_name': 'title', 'position': 'title', 'должность': 'title',
    'job_description': 'description', 'text': 'description', 'body': 'description', 'описание': 'description',
    'company': 'company_profile', 'company_name': 'company_profile', 'employer': 'company_profile', 'organization': 'company_profile', 'работодатель': 'company_profile',
    'city': 'location', 'region': 'location', 'place': 'location', 'город': 'location', 'адрес': 'location',
    'salary': 'salary_range', 'wage': 'salary_range', 'зарплата': 'salary_range',
    'требования': 'requirements', 'requirements_text': 'requirements',
    'условия': 'benefits', 'benefits_text': 'benefits',
    'url': 'link', 'href': 'link', 'job_url': 'link',
    'is_fake': 'fraudulent', 'label': 'fraudulent', 'fraud': 'fraudulent', 'is_fraudulent': 'fraudulent',
    'risk': 'risk_level',
}


def standardize_job_columns(raw_df):
    df = raw_df.copy()
    df.columns = [str(c).lower().strip() for c in df.columns]
    df = df.rename(columns={k: v for k, v in KZ_COL_MAP.items() if k in df.columns})

    if df.columns.duplicated().any():
        merged = pd.DataFrame(index=df.index)
        for col in pd.unique(df.columns):
            same = df.loc[:, df.columns == col]
            merged[col] = same.bfill(axis=1).iloc[:, 0] if same.shape[1] > 1 else same.iloc[:, 0]
        df = merged

    required_cols = ['title', 'description', 'company_profile', 'location', 'salary_range', 'requirements', 'benefits', 'link', 'industry']
    for col in required_cols:
        if col not in df.columns:
            df[col] = ''
        df[col] = df[col].fillna('').astype(str)
    return df


df = standardize_job_columns(raw_kz)
print(f'Standardised: {df.shape}')
print('Columns:', df.columns.tolist())
df.head(3)

Standardised: (636, 14)
Columns: ['title', 'company_profile', 'has_company_name', 'salary_range', 'has_salary', 'link', 'source', 'description', 'fraudulent', 'id', 'location', 'requirements', 'benefits', 'industry']


,title,company_profile,has_company_name,salary_range,has_salary,link,source,description,fraudulent,id,location,requirements,benefits,industry
0,Требуется водитель для экскаватора погрузчика ...,,0,,0,https://www.olx.kz/obyavlenie/rabota/trebuetsy...,olx.kz,требуется водитель для экскаватора погрузчика ...,1,458,,,,
1,Администратор в студию лазерной эпиляции,ТОО Лала,1,,0,https://almaty.hh.kz/vacancy/131831768?hhtmFro...,hh.kz,👩‍💼 администратор студии лазерной эпиляции | в...,0,281,,,,
2,Оператор call-центра,ТОО Медицинский центр EMIRMED,1,"от 200 000 до 350 000 ₸ за месяц, на руки",1,https://almaty.hh.kz/vacancy/131619531?hhtmFro...,hh.kz,многопрофильные медицинские центры эмирмед уже...,0,276,,,,


In [4]:
def is_safe_label(value):
    v = str(value).strip().lower()
    return v in {'0', 'false', 'real', 'safe', 'legit', 'not fraud', 'not_fraud'}

if 'fraudulent' in df.columns and df['fraudulent'].astype(str).str.strip().ne('').any():
    safe_jobs = df[df['fraudulent'].apply(is_safe_label)].copy()
    filter_method = 'fraudulent / label indicates safe job'
elif 'risk_level' in df.columns and df['risk_level'].astype(str).str.strip().ne('').any():
    safe_jobs = df[df['risk_level'].astype(str).str.lower().str.strip() != 'high'].copy()
    filter_method = 'risk_level != High'
else:
    safe_jobs = df.copy()
    filter_method = 'no labels - all jobs used (fallback)'
    print('No fraud labels found. Run the main predictor first for best results.')

safe_jobs = safe_jobs.reset_index(drop=True)
print(f'Filter method : {filter_method}')
print(f'Total jobs    : {len(df):,}')
print(f'Safe jobs     : {len(safe_jobs):,}  ({len(safe_jobs)/max(len(df),1):.1%})')

Filter method : fraudulent / label indicates safe job
Total jobs    : 636
Safe jobs     : 496  (78.0%)


---
## **5. Text Preparation**

In [5]:
def clean_for_rec(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z0-9\u0400-\u04FF\s\+\#]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def prepare_job_text(row) -> str:
    """Combine vacancy fields into one weighted text."""
    title = clean_for_rec(str(row.get('title', '')))
    company = clean_for_rec(str(row.get('company_profile', '')))
    desc = clean_for_rec(str(row.get('description', '')))
    req = clean_for_rec(str(row.get('requirements', '')))
    ben = clean_for_rec(str(row.get('benefits', '')))
    industry = clean_for_rec(str(row.get('industry', '')))
    parts = ([title] * 5) + ([req] * 3) + [industry, company, desc, ben]
    return ' '.join(p for p in parts if p)

if len(safe_jobs) > 0:
    sample = prepare_job_text(safe_jobs.iloc[0])
    print('Sample text (150 chars):', sample[:150])
    avg_len = safe_jobs.apply(prepare_job_text, axis=1).str.len().mean()
    print(f'Avg prepared text length: {avg_len:.0f} chars')

Sample text (150 chars): администратор в студию лазерной эпиляции администратор в студию лазерной эпиляции администратор в студию лазерной эпиляции администратор в студию лазе
Avg prepared text length: 670 chars


---
## **5.1. Extended Job Family Detection**

This section fixes irrelevant recommendations such as **backend developer → waiter** or **administrator → taxi driver**.

Instead of searching across all safe jobs, the system first detects the professional category (`job_family`) and then recommends jobs inside the same or related family.


In [6]:
JOB_FAMILY_KEYWORDS = {
    'tech_backend_frontend': ['backend','back end','frontend','front end','fullstack','full stack','developer','software engineer','программист','разработчик','веб разработчик','web developer','python','java','javascript','typescript','react','vue','angular','django','fastapi','spring','node','rest api','api','postgresql','mysql','docker','git','linux','devops','software','web application'],
    'data_ai': ['data scientist','data analyst','machine learning','ml engineer','ai engineer','аналитик данных','дата аналитик','data science','artificial intelligence','искусственный интеллект','скоринг','modeling','модель','python','pandas','numpy','sklearn','pytorch','tensorflow','bert','nlp','llm','sql','power bi','powerbi','tableau','big data','spark','hadoop','аналитик'],
    'it_support_sysadmin': ['system administrator','sysadmin','it support','technical support','helpdesk','администратор linux','системный администратор','техническая поддержка','поддержка пользователей','сетевой инженер','network engineer','windows server','linux','active directory','1c','1с','настройка компьютеров','ремонт компьютеров','сервисный инженер'],
    'admin_office_reception': ['administrator','administrative assistant','office manager','receptionist','secretary','администратор','офис менеджер','офис-менеджер','секретарь','ресепшен','reception','прием звонков','документооборот','документы','делопроизводство','excel','word','schedule','scheduling','appointments','встреча гостей','запись клиентов','корреспонденция','ассистент руководителя'],
    'driver_transport': ['driver','taxi','personal driver','водитель','такси','шофер','персональный водитель','автобус','грузовик','truck driver','экскаватор','погрузчик','категория b','категория c','категория d','водительские права','пассажирские перевозки','доставка пассажиров','route planning','navigation','безопасное вождение'],
    'courier_delivery': ['courier','delivery','курьер','доставка','доставщик','пеший курьер','велокурьер','яндекс доставка','glovo','wolt','развоз','посылки','заказы','delivery driver','food delivery'],
    'sales_retail_cashier': ['sales','sales manager','seller','retail','cashier','consultant','продавец','продавец консультант','кассир','менеджер по продажам','розница','магазин','бутик','торговый представитель','активные продажи','b2b','b2c','клиенты','консультация покупателей','мерчендайзер','promoter','промоутер'],
    'call_center_support': ['call center','contact center','operator','customer support','client support','оператор','колл центр','call-центр','контакт центр','поддержка клиентов','клиентская поддержка','переписка с клиентами','чат оператор','оператор чата','телефонные звонки','исходящие звонки','входящие звонки','customer service'],
    'marketing_smm': ['marketing','marketer','smm','social media','content manager','таргетолог','маркетолог','смм','контент менеджер','копирайтер','seo','instagram','tiktok','facebook','ads','реклама','таргетированная реклама','бренд','продвижение'],
    'accounting_finance': ['accountant','finance','financial analyst','бухгалтер','кассир бухгалтер','финансист','финансовый аналитик','учет','1c','1с бухгалтерия','налоги','налоговая отчетность','зарплата','банк','кредит','аудит','экономист','платежи','invoice'],
    'hr_recruiting': ['hr','human resources','recruiter','recruiting','talent acquisition','кадровик','hr manager','эйчар','рекрутер','подбор персонала','кадры','персонал','адаптация сотрудников','интервью','собеседование'],
    'hospitality_food': ['waiter','waitress','barista','cook','chef','restaurant','cafe','hotel','официант','официантка','бариста','повар','шеф повар','ресторан','кафе','гостиница','отель','хостес','посудомойщик','кухня','сушист','пиццмейкер','бармен','barman'],
    'beauty_wellness': ['beauty','salon','hairdresser','nail','makeup','cosmetologist','массажист','парикмахер','мастер маникюра','маникюр','педикюр','визажист','косметолог','бровист','ресницы','салон красоты','барбер','spa','фитнес тренер','тренер'],
    'medical_pharma': ['doctor','nurse','pharmacist','medical','clinic','врач','медсестра','медицинская сестра','фармацевт','провизор','клиника','медицина','лаборант','стоматолог','педиатр','терапевт','медицинский представитель'],
    'cleaning_housekeeping': ['cleaner','cleaning','housekeeping','уборщик','уборщица','клининг','горничная','уборка','мойщик','дворник','домработница','помощница по дому','санитарка'],
    'security_guard': ['security','guard','охранник','охрана','сторож','security guard','безопасность','контроль доступа','видеонаблюдение'],
    'construction_engineering': ['engineer','construction','architect','civil engineer','инженер','строитель','строительство','архитектор','прораб','сметчик','электрик','сантехник','монтажник','сварщик','мастер участка','autocad','проектировщик','техник'],
    'warehouse_logistics': ['warehouse','logistics','loader','storekeeper','кладовщик','склад','грузчик','логист','комплектовщик','упаковщик','сборщик заказов','оператор склада','inventory','wms','forklift','карщик'],
    'education_training': ['teacher','tutor','trainer','education','преподаватель','учитель','репетитор','тренер преподаватель','обучение','курсы','школа','детский сад','воспитатель','методист','english teacher','математика'],
    'design_creative_media': ['designer','graphic designer','ui ux','ux ui','illustrator','video editor','motion designer','дизайнер','графический дизайнер','видеомонтаж','монтажер','фотограф','оператор','креатив','figma','photoshop','illustrator','canva','media'],
    'legal_compliance': ['lawyer','legal','compliance','юрист','юрисконсульт','право','договоры','суд','комплаенс','aml','kyc','регулятор','нормативные документы'],
    'management_executive': ['manager','project manager','product manager','director','head of','руководитель','директор','управляющий','менеджер проекта','проджект менеджер','продукт менеджер','team lead','тимлид','начальник отдела','координатор проекта'],
    'manufacturing_bluecollar': ['operator','production','factory','worker','оператор производства','рабочий','производство','завод','станок','упаковщик','сборщик','швея','мастер производства','токарь','слесарь','механик','техник производства'],
    'agriculture_farm': ['farm','agriculture','агроном','ферма','сельское хозяйство','животновод','тракторист','садовник','теплица','урожай'],
    'domestic_care': ['nanny','caregiver','babysitter','сиделка','няня','уход','помощница','домработница','elderly care','child care'],
}

RELATED_FAMILIES = {
    'tech_backend_frontend': ['data_ai', 'it_support_sysadmin'],
    'data_ai': ['tech_backend_frontend', 'it_support_sysadmin', 'accounting_finance'],
    'it_support_sysadmin': ['tech_backend_frontend', 'data_ai'],
    'admin_office_reception': ['call_center_support', 'hr_recruiting', 'management_executive'],
    'driver_transport': ['courier_delivery', 'warehouse_logistics'],
    'courier_delivery': ['driver_transport', 'warehouse_logistics'],
    'sales_retail_cashier': ['call_center_support', 'marketing_smm'],
    'call_center_support': ['sales_retail_cashier', 'admin_office_reception'],
    'marketing_smm': ['sales_retail_cashier', 'design_creative_media'],
    'accounting_finance': ['legal_compliance', 'admin_office_reception', 'data_ai'],
    'hr_recruiting': ['admin_office_reception', 'management_executive'],
    'hospitality_food': ['sales_retail_cashier', 'cleaning_housekeeping'],
    'beauty_wellness': ['sales_retail_cashier', 'hospitality_food'],
    'medical_pharma': ['admin_office_reception', 'sales_retail_cashier'],
    'cleaning_housekeeping': ['hospitality_food', 'domestic_care'],
    'security_guard': ['driver_transport', 'warehouse_logistics'],
    'construction_engineering': ['manufacturing_bluecollar'],
    'warehouse_logistics': ['driver_transport', 'courier_delivery', 'manufacturing_bluecollar'],
    'education_training': ['admin_office_reception'],
    'design_creative_media': ['marketing_smm'],
    'legal_compliance': ['accounting_finance'],
    'management_executive': ['admin_office_reception', 'hr_recruiting'],
    'manufacturing_bluecollar': ['construction_engineering', 'warehouse_logistics'],
    'agriculture_farm': ['manufacturing_bluecollar'],
    'domestic_care': ['cleaning_housekeeping'],
}


def detect_job_family(text: str) -> str:
    text = clean_for_rec(text)
    if not text:
        return 'unknown'
    scores = {}
    for family, keywords in JOB_FAMILY_KEYWORDS.items():
        score = 0
        for kw in keywords:
            kw_clean = clean_for_rec(kw)
            if kw_clean and kw_clean in text:
                score += 2 if ' ' in kw_clean else 1
        if score > 0:
            scores[family] = score
    return max(scores, key=scores.get) if scores else 'unknown'

safe_jobs['job_family'] = safe_jobs.apply(lambda row: detect_job_family(prepare_job_text(row)), axis=1)
print('Top job families in safe_jobs:')
print(safe_jobs['job_family'].value_counts().head(25))

Top job families in safe_jobs:
job_family
unknown                     80
driver_transport            63
admin_office_reception      50
sales_retail_cashier        45
hospitality_food            28
construction_engineering    28
accounting_finance          26
warehouse_logistics         22
hr_recruiting               16
cleaning_housekeeping       16
tech_backend_frontend       15
call_center_support         14
manufacturing_bluecollar    14
courier_delivery            13
security_guard              11
education_training           9
medical_pharma               8
it_support_sysadmin          8
marketing_smm                6
design_creative_media        5
beauty_wellness              5
domestic_care                4
data_ai                      4
legal_compliance             3
management_executive         2
Name: count, dtype: int64


In [7]:
# Load additional real HH jobs and standardize them the same way.
EXTRA_DATA_PATH = '/Users/aruzhantleukul/Desktop/machine learning/TrustJob/real_hh_data.csv'


if EXTRA_DATA_PATH:
    extra_raw = read_jobs_csv(EXTRA_DATA_PATH)
    extra = standardize_job_columns(extra_raw)
    if 'fraudulent' in extra.columns and extra['fraudulent'].astype(str).str.strip().ne('').any():
        extra_safe = extra[extra['fraudulent'].apply(is_safe_label)].copy()
    else:
        extra_safe = extra.copy()

    for col in safe_jobs.columns:
        if col not in extra_safe.columns:
            extra_safe[col] = ''
    for col in extra_safe.columns:
        if col not in safe_jobs.columns:
            safe_jobs[col] = ''

    safe_jobs = pd.concat([safe_jobs, extra_safe[safe_jobs.columns]], ignore_index=True).fillna('')
    safe_jobs['job_family'] = safe_jobs.apply(lambda row: detect_job_family(prepare_job_text(row)), axis=1)
    print(f'Added extra dataset: {EXTRA_DATA_PATH}')
else:
    print('Extra dataset real_hh_data.csv not found. Continuing with Kazakhstan dataset only.')

safe_jobs = safe_jobs.reset_index(drop=True)
print(f'Dataset after optional extra jobs: {len(safe_jobs):,} rows')
print(safe_jobs['job_family'].value_counts().head(25))

Added extra dataset: /Users/aruzhantleukul/Desktop/machine learning/TrustJob/real_hh_data.csv
Dataset after optional extra jobs: 496 rows
job_family
unknown                     80
driver_transport            63
admin_office_reception      50
sales_retail_cashier        45
hospitality_food            28
construction_engineering    28
accounting_finance          26
warehouse_logistics         22
hr_recruiting               16
cleaning_housekeeping       16
tech_backend_frontend       15
call_center_support         14
manufacturing_bluecollar    14
courier_delivery            13
security_guard              11
education_training           9
medical_pharma               8
it_support_sysadmin          8
marketing_smm                6
design_creative_media        5
beauty_wellness              5
domestic_care                4
data_ai                      4
legal_compliance             3
management_executive         2
Name: count, dtype: int64


---
## **6. Build Recommendation Index**

We fit TF-IDF **only on safe jobs** and precompute a vector matrix.  
At query time: one `transform()` call + cosine similarity - fast, no retraining needed.


In [8]:
def build_recommendation_index(safe_jobs: pd.DataFrame):
    print(f'Building index for {len(safe_jobs):,} safe jobs...')
    job_texts = safe_jobs.apply(prepare_job_text, axis=1).tolist()
    vectorizer = TfidfVectorizer(
        max_features=30_000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=1,
        max_df=0.95,
        strip_accents='unicode'
    )
    job_vectors = vectorizer.fit_transform(job_texts)
    print('   Index built.')
    print(f'   Vocabulary : {len(vectorizer.vocabulary_):,} tokens')
    print(f'   Matrix     : {job_vectors.shape}')
    return vectorizer, job_vectors, job_texts

rec_vectorizer, job_vectors, job_texts = build_recommendation_index(safe_jobs)

Building index for 496 safe jobs...
   Index built.
   Vocabulary : 30,000 tokens
   Matrix     : (496, 30000)


---
## **7. Similar Job Recommendations**

Given any job posting (real or flagged fake), find the most similar **safe** alternatives.


In [9]:
def get_output_columns(df, score_col):
    preferred = ['rank', 'title', 'company_profile', 'location', 'salary_range', 'link', 'job_family', score_col]
    return [col for col in preferred if col in df.columns]


def get_candidate_jobs_by_family(query_family, safe_jobs, top_n=5, allow_related=True):
    if 'job_family' not in safe_jobs.columns:
        safe_jobs = safe_jobs.copy()
        safe_jobs['job_family'] = safe_jobs.apply(lambda row: detect_job_family(prepare_job_text(row)), axis=1)
    if query_family == 'unknown':
        return safe_jobs.copy(), 'unknown family: all safe jobs used'
    families = [query_family]
    candidate_jobs = safe_jobs[safe_jobs['job_family'].isin(families)].copy()
    search_scope = f'exact family only: {families}'
    if allow_related and len(candidate_jobs) < top_n:
        families = [query_family] + RELATED_FAMILIES.get(query_family, [])
        candidate_jobs = safe_jobs[safe_jobs['job_family'].isin(families)].copy()
        search_scope = f'exact + related families: {families}'
    return candidate_jobs, search_scope


def recommend_similar_jobs(input_job: dict, safe_jobs: pd.DataFrame, vectorizer, job_vectors,
                           top_n: int = 5, min_score: float = 0.03, allow_related: bool = True) -> pd.DataFrame:
    text = prepare_job_text(pd.Series(input_job))
    if not text.strip():
        return pd.DataFrame()
    query_family = detect_job_family(text)
    candidate_jobs, search_scope = get_candidate_jobs_by_family(query_family, safe_jobs, top_n, allow_related)
    if len(candidate_jobs) == 0:
        print(f'Detected job family: {query_family}')
        print('No relevant jobs found for this family.')
        return pd.DataFrame()
    candidate_indices = candidate_jobs.index.to_numpy()
    scores = cosine_similarity(vectorizer.transform([text]), job_vectors[candidate_indices]).flatten()
    candidate_jobs = candidate_jobs.copy()
    candidate_jobs['similarity_score'] = scores
    candidate_jobs = candidate_jobs[candidate_jobs['similarity_score'] >= min_score]
    candidate_jobs = candidate_jobs.sort_values('similarity_score', ascending=False).head(top_n)
    candidate_jobs.insert(0, 'rank', range(1, len(candidate_jobs) + 1))
    candidate_jobs['similarity_score'] = candidate_jobs['similarity_score'].round(4)
    print(f'Detected job family: {query_family}')
    print(f'Search scope: {search_scope}')
    print(f'Found jobs after filtering: {len(candidate_jobs)}')
    return candidate_jobs[get_output_columns(candidate_jobs, 'similarity_score')].reset_index(drop=True)

In [10]:
sample_job = {
    'title'       : 'Python Backend Developer',
    'description' : 'Build REST APIs and work with databases in a fintech startup.',
    'requirements': 'Python, FastAPI, PostgreSQL, Docker, Git, 2+ years.',
    'benefits'    : 'Remote, competitive salary, health insurance.',
    'location'    : 'Almaty',
    'salary_range': '500000-800000 KZT',
}

similar = recommend_similar_jobs(sample_job, safe_jobs, rec_vectorizer, job_vectors, top_n=5)
print('Top 5 similar safe jobs:')
print(similar.to_string(index=False) if not similar.empty else '  No results.')

Detected job family: tech_backend_frontend
Search scope: exact family only: ['tech_backend_frontend']
Found jobs after filtering: 5
Top 5 similar safe jobs:
 rank                              title               company_profile location                             salary_range                                                                link            job_family  similarity_score
    1 PHP Developer (Laravel / Asterisk) ИП Елисеев Максим Анатольевич                         2 000 $ за месяц, на руки https://almaty.hh.kz/vacancy/132199457?hhtmFrom=vacancy_search_list tech_backend_frontend            0.0902
    2         Backend-разработчик (Java)                 ТОО Yurt Tech                                                   https://almaty.hh.kz/vacancy/132180241?hhtmFrom=vacancy_search_list tech_backend_frontend            0.0771
    3          QA инженер Middle/Middle+               ТОО My Startups                                                   https://almaty.hh.kz/vacancy/132118

---
## 8. Resume-to-Job Matching

The user pastes their resume. We find the best-matching safe jobs from the Kazakhstan database.


In [11]:
def match_resume_to_jobs(resume_text: str, safe_jobs: pd.DataFrame, vectorizer, job_vectors,
                         top_n: int = 5, min_score: float = 0.03, allow_related: bool = True) -> pd.DataFrame:
    """
    Match a resume to safe jobs.
    1. Detect resume job_family.
    2. Search only inside the same/related family.
    3. Apply TF-IDF + cosine similarity.
    4. Remove weak matches with min_score.
    """
    if not isinstance(resume_text, str) or len(resume_text.strip()) < 10:
        print('Resume too short.')
        return pd.DataFrame()
    cleaned_resume = clean_for_rec(resume_text)
    resume_family = detect_job_family(cleaned_resume)
    candidate_jobs, search_scope = get_candidate_jobs_by_family(resume_family, safe_jobs, top_n, allow_related)
    if len(candidate_jobs) == 0:
        print(f'Detected resume family: {resume_family}')
        print('No relevant jobs found for this family.')
        return pd.DataFrame()
    candidate_indices = candidate_jobs.index.to_numpy()
    scores = cosine_similarity(vectorizer.transform([cleaned_resume]), job_vectors[candidate_indices]).flatten()
    candidate_jobs = candidate_jobs.copy()
    candidate_jobs['match_score'] = scores
    candidate_jobs = candidate_jobs[candidate_jobs['match_score'] >= min_score]
    candidate_jobs = candidate_jobs.sort_values('match_score', ascending=False).head(top_n)
    candidate_jobs.insert(0, 'rank', range(1, len(candidate_jobs) + 1))
    candidate_jobs['match_score'] = candidate_jobs['match_score'].round(4)
    print(f'Detected resume family: {resume_family}')
    print(f'Search scope: {search_scope}')
    print(f'Found jobs after filtering: {len(candidate_jobs)}')
    return candidate_jobs[get_output_columns(candidate_jobs, 'match_score')].reset_index(drop=True)

In [12]:
sample_resume = """
Software engineer with 4 years of Python development.
Skilled in FastAPI, Django, PostgreSQL, Redis, Docker, Kubernetes.
Led backend team at fintech startup in Almaty.
Bachelor in Computer Science, KBTU. Python, SQL, JavaScript.
Looking for remote-friendly backend position in Kazakhstan.
"""

matches = match_resume_to_jobs(sample_resume, safe_jobs, rec_vectorizer, job_vectors, top_n=5, min_score=0.03)
print('Top 5 resume matches:')
print(matches.to_string(index=False) if not matches.empty else '  No results.')

Detected resume family: tech_backend_frontend
Search scope: exact family only: ['tech_backend_frontend']
Found jobs after filtering: 4
Top 5 resume matches:
 rank                      title  company_profile location                             salary_range                                                                link            job_family  match_score
    1 Backend-разработчик (Java)    ТОО Yurt Tech                                                   https://almaty.hh.kz/vacancy/132180241?hhtmFrom=vacancy_search_list tech_backend_frontend       0.1000
    2 Manual QA инженер (Middle)   ТОО A1 CAPITAL                    от 250 000 ₸ за месяц, на руки https://almaty.hh.kz/vacancy/132151566?hhtmFrom=vacancy_search_list tech_backend_frontend       0.0568
    3         Стажер-разработчик ТОО Prime Source          от 200 000 до 250 000 ₸ за месяц на руки https://almaty.hh.kz/vacancy/131896635?hhtmFrom=vacancy_search_list tech_backend_frontend       0.0519
    4  QA инженер Middle/Middle

---
## **8.1. Extra Resume Matching Tests**

These tests prove that the recommender does not mix unrelated professions.

Expected behavior:
- backend resume → IT/backend jobs
- administrator resume → office/admin jobs
- taxi driver resume → driver/transport jobs


In [13]:
admin_resume = """
Office administrator with experience in daily office operations, reception,
answering phone calls, greeting clients, scheduling appointments, preparing documents,
working with Microsoft Excel and Word, organizing meetings, managing correspondence,
and supporting managers with administrative tasks.
Skills: administration, office management, reception, documents, Excel, Word,
scheduling, communication, reporting, customer service.
"""

driver_resume = """
Professional taxi driver with experience in passenger transportation,
safe driving, route planning, navigation apps, traffic rules, vehicle maintenance,
city roads, car cleanliness, and polite customer service.
Skills: driver, taxi, driving, passenger transport, navigation, road safety,
vehicle maintenance, route planning, customer service.
"""

print('\nADMIN RESUME TEST')
admin_matches = match_resume_to_jobs(admin_resume, safe_jobs, rec_vectorizer, job_vectors, top_n=5, min_score=0.03)
print(admin_matches.to_string(index=False) if not admin_matches.empty else '  No results.')

print('\nDRIVER RESUME TEST')
driver_matches = match_resume_to_jobs(driver_resume, safe_jobs, rec_vectorizer, job_vectors, top_n=5, min_score=0.03)
print(driver_matches.to_string(index=False) if not driver_matches.empty else '  No results.')


ADMIN RESUME TEST
Detected resume family: admin_office_reception
Search scope: exact family only: ['admin_office_reception']
Found jobs after filtering: 4
 rank                        title                                             company_profile location                         salary_range                                                                link             job_family  match_score
    1               Adidas Trainee                                           adidas Kazakhstan                                               https://almaty.hh.kz/vacancy/131950221?hhtmFrom=vacancy_search_list admin_office_reception       0.1395
    2 PROJECT COORDINATION SUPPORT ФИЛИАЛ АССОЦИАЦИИ ВРАЧИ БЕЗ ГРАНИЦ (ШВЕЙЦАРИЯ) В КАЗАХСТАНЕ          750 000 ₸ за месяц до вычета налогов https://almaty.hh.kz/vacancy/132146486?hhtmFrom=vacancy_search_list admin_office_reception       0.1371
    3  Document Control Specialist                          ТОО ZTE Kazakhstan (ЗТИ Казахстан)               

---
## **9. Skill Gap Analysis**

After finding the best job match, show the user which skills they are **missing** based on their resume vs the job requirements.

**Method:** Extract skills from both texts using a curated bilingual keyword list, then compare the two sets.


In [14]:
SKILL_KEYWORDS = {
    # Programming / IT
    'python','java','javascript','typescript','c++','c#','go','golang','rust','kotlin','swift','ruby','php','scala','r',
    'django','fastapi','flask','react','vue','angular','nextjs','spring','express','laravel','rails','nodejs',
    'postgresql','mysql','mongodb','redis','elasticsearch','sqlite','oracle','docker','kubernetes','aws','azure','gcp',
    'terraform','jenkins','linux','nginx','git','rest api','graphql','microservices','agile','scrum','sql','nosql',
    'pandas','numpy','sklearn','tensorflow','pytorch','spark','hadoop','airflow','dbt','tableau','powerbi','power bi',
    'figma','photoshop','canva','excel','word','ms office','google workspace',
    # Admin / office
    'administration','office management','reception','document preparation','scheduling','reporting','customer service',
    'communication','correspondence','phone calls','appointments','администрирование','документооборот','секретарь',
    'ресепшен','прием звонков','работа с документами',
    # Driver / logistics
    'driving','driver','taxi','navigation','route planning','road safety','vehicle maintenance','passenger transport',
    'delivery','courier','warehouse','logistics','inventory','wms','водитель','такси','доставка','курьер','склад','логистика',
    # Sales / support / marketing
    'sales','retail','cashier','negotiation','crm','cold calls','lead generation','smm','marketing','seo','copywriting',
    'call center','client support','продажи','кассир','клиенты','маркетинг','смм','оператор','колл центр',
    # Finance / legal / HR
    'accounting','finance','audit','tax','1c','1с','recruiting','hr','interviewing','onboarding','legal','compliance','kyc','aml',
    'бухгалтерия','финансы','налоги','рекрутинг','кадры','юрист','комплаенс',
    # Other domains
    'cooking','barista','waiter','cleaning','security','construction','autocad','teaching','tutoring','nursing','pharmacy',
    'повар','бариста','официант','уборка','охранник','строительство','учитель','репетитор','медсестра','фармацевт',
}


def extract_skills(text: str) -> set:
    tl = clean_for_rec(text)
    return {s for s in SKILL_KEYWORDS if clean_for_rec(s) in tl}


def skill_gap_analysis(resume_text: str, job_text: str) -> dict:
    resume_skills = extract_skills(resume_text)
    job_skills = extract_skills(job_text)
    if not job_skills:
        return {'resume_skills': resume_skills, 'job_skills': set(), 'matched_skills': resume_skills,
                'missing_skills': set(), 'extra_skills': resume_skills, 'match_percentage': 100.0,
                'gap_severity': 'Low', 'recommendation': 'No specific skills detected in job posting.'}
    matched = resume_skills & job_skills
    missing = job_skills - resume_skills
    extra = resume_skills - job_skills
    match_pct = round(len(matched) / len(job_skills) * 100, 1)
    if match_pct >= 70:
        sev = 'Low'
        advice = 'Strong match! You meet most requirements for this role.'
    elif match_pct >= 40:
        sev = 'Medium'
        advice = f'Good foundation. Consider learning: {", ".join(sorted(missing)[:5])}.'
    else:
        sev = 'High'
        advice = f'Significant gap. Focus first on: {", ".join(sorted(missing)[:5])}.'
    return {'resume_skills': resume_skills, 'job_skills': job_skills, 'matched_skills': matched,
            'missing_skills': missing, 'extra_skills': extra, 'match_percentage': match_pct,
            'gap_severity': sev, 'recommendation': advice}

In [15]:
sample_job_text = """
Senior Python Developer. Must have: FastAPI, PostgreSQL, Docker,
Redis, Kubernetes, AWS. Git required. Agile/Scrum. Microservices.
"""

gap = skill_gap_analysis(sample_resume, sample_job_text)

print('Skill Gap Analysis')
print(f'Match       : {gap["match_percentage"]}%')
print(f'Severity    : {gap["gap_severity"]}')
print(f'You have    : {sorted(gap["matched_skills"])}')
print(f'Missing     : {sorted(gap["missing_skills"])}')
print(f'Bonus       : {sorted(gap["extra_skills"])}')
print(f'Advice      : {gap["recommendation"]}')


── Skill Gap Analysis ──────────────────────────────────────────
Match       : 61.5%
Severity    : Medium
You have    : ['docker', 'fastapi', 'kubernetes', 'postgresql', 'python', 'r', 'redis', 'sql']
Missing     : ['agile', 'aws', 'git', 'microservices', 'scrum']
Bonus       : ['django', 'go', 'java', 'javascript']
Advice      : Good foundation. Consider learning: agile, aws, git, microservices, scrum.


---
## 10. Save Artifacts

In [16]:
joblib.dump(rec_vectorizer, 'models/recommendation_vectorizer.pkl')
print('models/recommendation_vectorizer.pkl')
joblib.dump(job_vectors, 'models/job_vectors.pkl')
print('models/job_vectors.pkl')
safe_jobs.to_csv('models/safe_kz_jobs.csv', index=False, encoding='utf-8-sig')
print('models/safe_kz_jobs.csv')
with open('models/skill_keywords.json', 'w', encoding='utf-8') as f:
    json.dump(sorted(SKILL_KEYWORDS), f, indent=2, ensure_ascii=False)
print('models/skill_keywords.json')
with open('models/job_family_keywords.json', 'w', encoding='utf-8') as f:
    json.dump(JOB_FAMILY_KEYWORDS, f, indent=2, ensure_ascii=False)
print('models/job_family_keywords.json')
with open('models/related_families.json', 'w', encoding='utf-8') as f:
    json.dump(RELATED_FAMILIES, f, indent=2, ensure_ascii=False)
print('models/related_families.json')
meta = {'n_safe_jobs': len(safe_jobs), 'vocab_size': len(rec_vectorizer.vocabulary_), 'filter_method': filter_method,
        'data_source': DATA_SOURCE, 'ngram_range': [1, 2], 'max_features': 30000,
        'job_families': sorted(safe_jobs['job_family'].unique().tolist())}
with open('models/recommendation_config.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)
print('models/recommendation_config.json')
print(f'\nAll artifacts saved. Safe jobs: {len(safe_jobs):,}')

models/recommendation_vectorizer.pkl
models/job_vectors.pkl
models/safe_kz_jobs.csv
models/skill_keywords.json
models/job_family_keywords.json
models/related_families.json
models/recommendation_config.json

All artifacts saved. Safe jobs: 496


---
## **11. Streamlit-Ready Module — `recommendation_engine.py`**

This file is saved to `models/recommendation_engine.py` and imported directly by the website.


In [18]:
ENGINE_CODE = """
# recommendation_engine.py — TrustJob KZ
# from models.recommendation_engine import get_similar_jobs, get_resume_matches, get_skill_gap

import re, json, warnings, joblib
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
warnings.filterwarnings("ignore")

_VEC = joblib.load("models/recommendation_vectorizer.pkl")
_VECTORS = joblib.load("models/job_vectors.pkl")
_JOBS = pd.read_csv("models/safe_kz_jobs.csv", encoding="utf-8-sig").fillna("").reset_index(drop=True)

with open("models/skill_keywords.json", encoding="utf-8") as f:
    _SKILLS = set(json.load(f))
with open("models/job_family_keywords.json", encoding="utf-8") as f:
    _JOB_FAMILY_KEYWORDS = json.load(f)
with open("models/related_families.json", encoding="utf-8") as f:
    _RELATED_FAMILIES = json.load(f)

def _clean(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"http\\S+|www\\.\\S+", " ", text)
    text = re.sub(r"[^a-z0-9\\u0400-\\u04FF\\s\\+\\#]", " ", text)
    return re.sub(r"\\s+", " ", text).strip()

def _prepare(row):
    t = _clean(str(row.get("title", "")))
    c = _clean(str(row.get("company_profile", "")))
    d = _clean(str(row.get("description", "")))
    r = _clean(str(row.get("requirements", "")))
    b = _clean(str(row.get("benefits", "")))
    industry = _clean(str(row.get("industry", "")))
    parts = ([t] * 5) + ([r] * 3) + [industry, c, d, b]
    return " ".join(p for p in parts if p)

def _detect_job_family(text):
    text = _clean(text)
    scores = {}
    for family, keywords in _JOB_FAMILY_KEYWORDS.items():
        score = 0
        for kw in keywords:
            kw_clean = _clean(kw)
            if kw_clean and kw_clean in text:
                score += 2 if " " in kw_clean else 1
        if score > 0:
            scores[family] = score
    return max(scores, key=scores.get) if scores else "unknown"

def _candidate_jobs(query_family, top_n=5, allow_related=True):
    if "job_family" not in _JOBS.columns or query_family == "unknown":
        return _JOBS.copy(), "unknown/all jobs"
    families = [query_family]
    candidates = _JOBS[_JOBS["job_family"].isin(families)].copy()
    if allow_related and len(candidates) < top_n:
        families = [query_family] + _RELATED_FAMILIES.get(query_family, [])
        candidates = _JOBS[_JOBS["job_family"].isin(families)].copy()
    return candidates, str(families)

def _result(candidates, scores, score_col, top_n, min_score):
    out = candidates.copy()
    out[score_col] = scores
    out = out[out[score_col] >= min_score].sort_values(score_col, ascending=False).head(top_n)
    out.insert(0, "rank", range(1, len(out) + 1))
    out[score_col] = out[score_col].round(4)
    cols = [c for c in ["rank", "title", "company_profile", "location", "salary_range", "link", "job_family", score_col] if c in out.columns]
    return out[cols].reset_index(drop=True)

def get_similar_jobs(input_job: dict, top_n: int = 5, min_score: float = 0.03) -> pd.DataFrame:
    text = _prepare(input_job)
    if not text.strip():
        return pd.DataFrame()
    family = _detect_job_family(text)
    candidates, _ = _candidate_jobs(family, top_n=top_n, allow_related=True)
    if len(candidates) == 0:
        return pd.DataFrame()
    idx = candidates.index.to_numpy()
    scores = cosine_similarity(_VEC.transform([text]), _VECTORS[idx]).flatten()
    return _result(candidates, scores, "similarity_score", top_n, min_score)

def get_resume_matches(resume_text: str, top_n: int = 5, min_score: float = 0.03) -> pd.DataFrame:
    if not isinstance(resume_text, str) or len(resume_text.strip()) < 10:
        return pd.DataFrame()
    text = _clean(resume_text)
    family = _detect_job_family(text)
    candidates, _ = _candidate_jobs(family, top_n=top_n, allow_related=True)
    if len(candidates) == 0:
        return pd.DataFrame()
    idx = candidates.index.to_numpy()
    scores = cosine_similarity(_VEC.transform([text]), _VECTORS[idx]).flatten()
    return _result(candidates, scores, "match_score", top_n, min_score)

def get_skill_gap(resume_text: str, job_text: str) -> dict:
    rs = {s for s in _SKILLS if _clean(s) in _clean(resume_text)}
    js = {s for s in _SKILLS if _clean(s) in _clean(job_text)}
    if not js:
        return {"resume_skills": rs, "job_skills": js, "matched_skills": rs, "missing_skills": set(),
                "extra_skills": rs, "match_percentage": 100.0, "gap_severity": "Low", "recommendation": "No skills detected."}
    matched = rs & js
    missing = js - rs
    match_pct = round(len(matched) / len(js) * 100, 1)
    if match_pct >= 70:
        sev, adv = "Low", "Strong match!"
    elif match_pct >= 40:
        sev, adv = "Medium", "Learn: " + ", ".join(sorted(missing)[:5])
    else:
        sev, adv = "High", "Focus on: " + ", ".join(sorted(missing)[:5])
    return {"resume_skills": rs, "job_skills": js, "matched_skills": matched, "missing_skills": missing,
            "extra_skills": rs - js, "match_percentage": match_pct, "gap_severity": sev, "recommendation": adv}
"""
with open('models/recommendation_engine.py', 'w', encoding='utf-8') as f:
    f.write(ENGINE_CODE)
print('models/recommendation_engine.py saved.')

models/recommendation_engine.py saved.


---
## **12. Streamlit Integration Snippet**

In [19]:
STREAMLIT_SNIPPET = """
import sys
sys.path.append("models")
from recommendation_engine import get_similar_jobs, get_resume_matches, get_skill_gap

st.header("Safe Job Recommendations")

tab1, tab2, tab3 = st.tabs(["Similar Jobs", "Resume Match", "Skill Gap"])

with tab1:
    st.subheader("Jobs similar to this posting")
    if st.button("Find similar safe jobs"):
        results = get_similar_jobs(current_job_dict, top_n=5)
        if results.empty:
            st.info("No similar jobs found in the database.")
        else:
            for _, row in results.iterrows():
                with st.expander(f"{row['rank']}. {row['title']} — {row['location']}"):
                    st.write(f"**Company:** {row['company_profile']}")
                    st.write(f"**Salary:** {row['salary_range']}")
                    st.write(f"**Family:** {row['job_family']}")
                    st.write(f"**Similarity:** {row['similarity_score']:.0%}")
                    if row["link"]:
                        st.markdown(f"[View job posting]({row['link']})")

with tab2:
    st.subheader("Find jobs that match your resume")
    resume_input = st.text_area("Paste your resume here", height=220)
    if st.button("Find matching jobs") and resume_input:
        matches = get_resume_matches(resume_input, top_n=5)
        if matches.empty:
            st.info("No matches found. Try adding more details about skills and experience.")
        else:
            st.dataframe(matches[["rank","title","location","salary_range","job_family","match_score"]], use_container_width=True)

with tab3:
    st.subheader("Skill gap between your resume and this job")
    if st.button("Analyse skill gap") and resume_input:
        job_text = current_job_dict.get("description","") + " " + current_job_dict.get("requirements","")
        gap = get_skill_gap(resume_input, job_text)
        col1, col2, col3 = st.columns(3)
        col1.metric("Match", f"{gap['match_percentage']}%")
        col2.metric("Gap Severity", gap["gap_severity"])
        col3.metric("Missing Skills", len(gap["missing_skills"]))
        if gap["missing_skills"]:
            st.warning("**Missing:** " + ", ".join(sorted(gap["missing_skills"])))
        if gap["matched_skills"]:
            st.success("**You have:** " + ", ".join(sorted(gap["matched_skills"])))
        st.info(gap["recommendation"])
"""
print(STREAMLIT_SNIPPET)


# ── app.py (Streamlit) — Recommendation section ──────────────────────────────
import sys
sys.path.append("models")
from recommendation_engine import get_similar_jobs, get_resume_matches, get_skill_gap

st.header("🔍 Safe Job Recommendations")

tab1, tab2, tab3 = st.tabs(["Similar Jobs", "Resume Match", "Skill Gap"])

with tab1:
    st.subheader("Jobs similar to this posting")
    if st.button("Find similar safe jobs"):
        results = get_similar_jobs(current_job_dict, top_n=5)
        if results.empty:
            st.info("No similar jobs found in the database.")
        else:
            for _, row in results.iterrows():
                with st.expander(f"{row['rank']}. {row['title']} — {row['location']}"):
                    st.write(f"**Company:** {row['company_profile']}")
                    st.write(f"**Salary:** {row['salary_range']}")
                    st.write(f"**Family:** {row['job_family']}")
                    st.write(f"**Similarity:** {row['similarity_score']:.0

---
## **13. End-to-End Test**

In [21]:
print("  END-TO-END TEST")

print("\n1. Similar Job Recommendations")
sim = recommend_similar_jobs(sample_job, safe_jobs, rec_vectorizer, job_vectors, top_n=3)
cols = [c for c in ["rank", "title", "location", "job_family", "similarity_score"] if c in sim.columns]
print(sim[cols].to_string(index=False) if not sim.empty else "  No results")

print("\n2. Resume Matching: backend")
mtch = match_resume_to_jobs(sample_resume, safe_jobs, rec_vectorizer, job_vectors, top_n=3)
cols = [c for c in ["rank", "title", "location", "job_family", "match_score"] if c in mtch.columns]
print(mtch[cols].to_string(index=False) if not mtch.empty else "  No results")

print("\n3. Resume Matching: admin")
admin_test = match_resume_to_jobs(admin_resume, safe_jobs, rec_vectorizer, job_vectors, top_n=3)
cols = [c for c in ["rank", "title", "location", "job_family", "match_score"] if c in admin_test.columns]
print(admin_test[cols].to_string(index=False) if not admin_test.empty else "  No results")

print("\n4. Resume Matching: driver/taxi")
driver_test = match_resume_to_jobs(driver_resume, safe_jobs, rec_vectorizer, job_vectors, top_n=3)
cols = [c for c in ["rank", "title", "location", "job_family", "match_score"] if c in driver_test.columns]
print(driver_test[cols].to_string(index=False) if not driver_test.empty else "  No results")

print("\n5. Skill Gap Analysis")
gap = skill_gap_analysis(sample_resume, sample_job_text)
print(f"  Match       : {gap['match_percentage']}%")
print(f"  Severity    : {gap['gap_severity']}")
print(f"  Missing     : {sorted(gap['missing_skills'])}")
print(f"  Advice      : {gap['recommendation']}")
print("\nAll functions working.")

  END-TO-END TEST

1. Similar Job Recommendations
Detected job family: tech_backend_frontend
Search scope: exact family only: ['tech_backend_frontend']
Found jobs after filtering: 3
 rank                              title location            job_family  similarity_score
    1 PHP Developer (Laravel / Asterisk)          tech_backend_frontend            0.0902
    2         Backend-разработчик (Java)          tech_backend_frontend            0.0771
    3          QA инженер Middle/Middle+          tech_backend_frontend            0.0540

2. Resume Matching: backend
Detected resume family: tech_backend_frontend
Search scope: exact family only: ['tech_backend_frontend']
Found jobs after filtering: 3
 rank                      title location            job_family  match_score
    1 Backend-разработчик (Java)          tech_backend_frontend       0.1000
    2 Manual QA инженер (Middle)          tech_backend_frontend       0.0568
    3         Стажер-разработчик          tech_backend_frontend

---
## **14. Summary**

### Artifacts saved to `models/`

| File | Purpose |
|---|---|
| `recommendation_vectorizer.pkl` | Fitted TF-IDF on safe jobs |
| `job_vectors.pkl` | Precomputed sparse matrix (n_jobs × vocab) |
| `safe_kz_jobs.csv` | Filtered safe job database |
| `skill_keywords.json` | Bilingual skill keyword list |
| `job_family_keywords.json` | Extended job family dictionary |
| `related_families.json` | Related professional categories for fallback search |
| `recommendation_config.json` | Index metadata |
| `recommendation_engine.py` | Standalone module for Streamlit/FastAPI |

### Functions summary

| Function | Input | Returns |
|---|---|---|
| `prepare_job_text(row)` | DataFrame row | Weighted combined text |
| `detect_job_family(text)` | Resume/job text | Professional category |
| `build_recommendation_index(df)` | Safe jobs DataFrame | vectorizer, matrix, texts |
| `recommend_similar_jobs(job, ...)` | Job dict | Top-N similar safe jobs |
| `match_resume_to_jobs(resume, ...)` | Resume string | Top-N matching jobs |
| `skill_gap_analysis(resume, job)` | Two strings | Gap dict with advice |

### Limitations

- Best results when resume and job database are in the same language. For Russian resumes matched against Russian job descriptions - works well. Cross-language matching benefits from translation first.
- Cosine similarity on TF-IDF is a keyword-overlap method. It does not understand semantic meaning (e.g. "engineer" vs "разработчик"). For production: replace with a multilingual sentence-transformer like `paraphrase-multilingual-mpnet-base-v2`.
- Skill gap analysis depends on the curated keyword list. Extend `SKILL_KEYWORDS` for non-tech domains.
- `job_family` is rule-based. It improves relevance and prevents obviously wrong matches, but it should be updated when new professions appear in the dataset.

### Next steps

1. Connect `get_similar_jobs()` to fraud detection — when a job is flagged fake, **auto-show** safe alternatives on the same page.
2. Add sentence-transformer embeddings for semantic (not just keyword) matching.
3. Collect user click feedback to re-rank recommendations over time.
4. Add user feedback buttons: “Relevant” / “Not relevant” for future learning-to-rank improvement.
